In [1]:
# Create a processed HAM10000 subset with 1 image per lesion_id
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
from tqdm import tqdm

In [2]:
PROJECT_ROOT = Path("..").resolve()

# Raw dataset location
HAM_RAW = PROJECT_ROOT / "data" / "raw" / "HAM10000"
META_PATH = HAM_RAW / "HAM10000_metadata.csv"
IMG1 = HAM_RAW / "ham10000_images_part_1"
IMG2 = HAM_RAW / "ham10000_images_part_2"

# Processed output location
HAM_PROCESSED = PROJECT_ROOT / "data" / "processed" / "HAM10000"
OUT_DATA = HAM_PROCESSED / "one_image_per_lesion"
OUT_IMAGES = OUT_DATA / "images"
OUT_META = OUT_DATA / "HAM10000_metadata_one_per_lesion.csv"
OUT_MAP = OUT_DATA / "lesion_to_image_mapping.csv"

OUT_IMAGES.mkdir(parents=True, exist_ok=True)

print("Reading metadata:", META_PATH)
print("Writing output to:", OUT_DATA)

Reading metadata: /Users/nicholasborch/Desktop/UNI/Bachelor Project/Repo/Bachelor-Project/data/raw/HAM10000/HAM10000_metadata.csv
Writing output to: /Users/nicholasborch/Desktop/UNI/Bachelor Project/Repo/Bachelor-Project/data/processed/HAM10000/one_image_per_lesion


In [3]:
# Load metadata + sample one image per lesion
df = pd.read_csv(META_PATH)

SEED = 42

# Choose ONE image per lesion_id (uniformly at random per group)
df_one = (
    df.groupby("lesion_id", group_keys=False)
      .apply(lambda g: g.sample(n=1, random_state=SEED))
      .reset_index(drop=True)
)

print("Original rows (images):", len(df))
print("Unique lesions:", df["lesion_id"].nunique())
print("New rows (1 per lesion):", len(df_one))
df_one.head()

Original rows (images): 10015
Unique lesions: 7470
New rows (1 per lesion): 7470


/var/folders/tc/p_2dw00d5zg0svx3_qj__8tc0000gn/T/ipykernel_71202/2865690592.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=SEED))


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000000,ISIC_0025346,nv,histo,60.0,male,back
1,HAM_0000001,ISIC_0027859,bkl,histo,70.0,female,face
2,HAM_0000002,ISIC_0033848,mel,histo,65.0,female,lower extremity
3,HAM_0000003,ISIC_0027886,nv,follow_up,55.0,male,trunk
4,HAM_0000004,ISIC_0024645,nv,follow_up,40.0,female,back


In [4]:
# Copy images
def find_image_path(image_id: str) -> Path | None:
    p1 = IMG1 / f"{image_id}.jpg"
    if p1.exists():
        return p1
    p2 = IMG2 / f"{image_id}.jpg"
    if p2.exists():
        return p2
    return None

missing = []
for img_id in tqdm(df_one["image_id"].tolist(), desc="Copying images"):
    src = find_image_path(img_id)
    if src is None:
        missing.append(img_id)
        continue
    dst = OUT_IMAGES / f"{img_id}.jpg"
    if not dst.exists():  # don't recopy if rerun
        shutil.copy2(src, dst)

print(f"Done. Missing images: {len(missing)}")
if missing:
    print("First missing ids:", missing[:10])

Copying images: 100%|██████████| 7470/7470 [00:14<00:00, 523.70it/s]

Done. Missing images: 0


In [5]:
# Save modified metadata (only selected images)
df_one.to_csv(OUT_META, index=False)

# Save mapping lesion_id -> chosen image_id
df_one[["lesion_id", "image_id", "dx"]].to_csv(OUT_MAP, index=False)

print("Saved:", OUT_META)
print("Saved:", OUT_MAP)
print("Images in output folder:", len(list(OUT_IMAGES.glob("*.jpg"))))

Saved: /Users/nicholasborch/Desktop/UNI/Bachelor Project/Repo/Bachelor-Project/data/processed/HAM10000/one_image_per_lesion/HAM10000_metadata_one_per_lesion.csv
Saved: /Users/nicholasborch/Desktop/UNI/Bachelor Project/Repo/Bachelor-Project/data/processed/HAM10000/one_image_per_lesion/lesion_to_image_mapping.csv
Images in output folder: 7470


In [6]:
# Sanity checks

# 1) Exactly one image per lesion
assert df_one["lesion_id"].nunique() == len(df_one), "Not exactly 1 row per lesion!"

# 2) All output images exist
missing_files = [img for img in df_one["image_id"] if not (OUT_IMAGES / f"{img}.jpg").exists()]
print("Missing copied files:", len(missing_files))

# 3) Compare class distributions (image-level vs lesion-level single-image)
print("\nClass counts (original):")
print(df["dx"].value_counts())

print("\nClass counts (one per lesion):")
print(df_one["dx"].value_counts())

Missing copied files: 0

Class counts (original):
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Class counts (one per lesion):
dx
nv       5403
bkl       727
mel       614
bcc       327
akiec     228
vasc       98
df         73
Name: count, dtype: int64
